# 6.0 Metadata Trial 001: sklearn Baselines

Purpose:
- Train metadata baseline models.
- Compare Logistic Regression and Random Forest.
- Evaluate on the existing stratified metadata test split.

Important:
The metadata dataset is contextual risk-factor data, not acute triage data.

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path
import sys

sys.path.insert(0, str(Path.cwd().parent))

from rural_stroke_assist.config import (
    METADATA_SPLIT_MANIFEST_PATH,
    METADATA_TRIAL_001_MODEL_PATH,
    METADATA_TRIAL_001_RESULTS_PATH,
    METADATA_TRIAL_001_PREDICTIONS_PATH,
    METADATA_TRIAL_001_CONFUSION_MATRIX_PATH,
)

from rural_stroke_assist.preprocessing.metadata import (
    prepare_metadata_modeling_frame,
    split_metadata_features_target,
)

from rural_stroke_assist.modeling.metadata_baselines import (
    build_logistic_regression_metadata_pipeline,
    build_random_forest_metadata_pipeline,
    save_metadata_model,
)

from rural_stroke_assist.modeling.evaluation import (
    evaluate_binary_classifier,
    create_classification_report_dict,
    save_confusion_matrix_plot,
    save_markdown_results,
)

In [2]:
metadata_df = pd.read_csv(METADATA_SPLIT_MANIFEST_PATH)

metadata_df.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke,split
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1,train
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1,train
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1,train
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1,train
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1,train


In [3]:
metadata_df.groupby(["split", "stroke"]).size()

split  stroke
test   0          729
       1           38
train  0         3403
       1          174
val    0          729
       1           37
dtype: int64

In [4]:
metadata_model_df = prepare_metadata_modeling_frame(metadata_df)

metadata_model_df.head()

,age,hypertension,heart_disease,avg_glucose_level,bmi,gender,ever_married,work_type,Residence_type,smoking_status,stroke,split
0,67.0,0,1,228.69,36.6,Male,Yes,Private,Urban,formerly smoked,1,train
1,61.0,0,0,202.21,NaN,Female,Yes,Self-employed,Rural,never smoked,1,train
2,80.0,0,1,105.92,32.5,Male,Yes,Private,Rural,never smoked,1,train
3,49.0,0,0,171.23,34.4,Female,Yes,Private,Urban,smokes,1,train
4,79.0,1,0,174.12,24.0,Female,Yes,Self-employed,Rural,never smoked,1,train


In [5]:
train_df = metadata_model_df[metadata_model_df["split"] == "train"].copy()
val_df = metadata_model_df[metadata_model_df["split"] == "val"].copy()
test_df = metadata_model_df[metadata_model_df["split"] == "test"].copy()

X_train, y_train = split_metadata_features_target(train_df)
X_val, y_val = split_metadata_features_target(val_df)
X_test, y_test = split_metadata_features_target(test_df)

len(train_df), len(val_df), len(test_df)

(3577, 766, 767)

In [6]:
logreg_model = build_logistic_regression_metadata_pipeline(random_state=42)

logreg_model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'hypertension',
                                                   'heart_disease',
                                                   'avg_glucose_level',
                                                   'bmi']),
                                                 ('categorical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['gender', 'ever_married',
                                                   'work_type',
                                                   'Residence_type',
                                                   'smoking_status'])])),
                ('model',
                 LogisticRegression(class_weight='balanced', max_iter=1000,
                                    random_state=42))])

In [7]:
logreg_val_pred = logreg_model.predict(X_val)
logreg_val_score = logreg_model.predict_proba(X_val)[:, 1]

logreg_val_metrics = evaluate_binary_classifier(
    y_true=y_val.to_numpy(),
    y_pred=logreg_val_pred,
    y_score=logreg_val_score,
)

logreg_val_metrics

{'accuracy': 0.737597911227154,
 'precision': 0.12727272727272726,
 'recall': 0.7567567567567568,
 'f1': 0.2178988326848249,
 'roc_auc': 0.8451785118451786}

In [8]:
rf_model = build_random_forest_metadata_pipeline(random_state=42)

rf_model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['age', 'hypertension',
                                                   'heart_disease',
                                                   'avg_glucose_level',
                                                   'bmi']),
                                                 ('categorical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['gender', 'ever_married',
                                                   'work_type',
                                                   'Residence_type',
                                                   'smoking_status'])])),
                ('model',
                 RandomForestClassifier(class_weight='balanced',
                                        n_estimators=300, n_jobs=-1,
                                        random_state=42))])

In [9]:
rf_val_pred = rf_model.predict(X_val)
rf_val_score = rf_model.predict_proba(X_val)[:, 1]

rf_val_metrics = evaluate_binary_classifier(
    y_true=y_val.to_numpy(),
    y_pred=rf_val_pred,
    y_score=rf_val_score,
)

rf_val_metrics

{'accuracy': 0.9516971279373369,
 'precision': 0.0,
 'recall': 0.0,
 'f1': 0.0,
 'roc_auc': 0.8163904645386127}

In [10]:
candidate_results = pd.DataFrame(
    [
        {"model_name": "logistic_regression", **logreg_val_metrics},
        {"model_name": "random_forest", **rf_val_metrics},
    ]
)

candidate_results

,model_name,accuracy,precision,recall,f1,roc_auc
0,logistic_regression,0.737598,0.127273,0.756757,0.217899,0.845179
1,random_forest,0.951697,0.000000,0.000000,0.000000,0.816390


In [11]:
best_row = candidate_results.sort_values(
    by=["f1", "roc_auc"],
    ascending=False,
).iloc[0]

best_model_name = best_row["model_name"]

if best_model_name == "logistic_regression":
    best_model = logreg_model
else:
    best_model = rf_model

best_model_name

'logistic_regression'

In [12]:
test_pred = best_model.predict(X_test)
test_score = best_model.predict_proba(X_test)[:, 1]

test_metrics = evaluate_binary_classifier(
    y_true=y_test.to_numpy(),
    y_pred=test_pred,
    y_score=test_score,
)

test_metrics

{'accuracy': 0.7235984354628422,
 'precision': 0.13135593220338984,
 'recall': 0.8157894736842105,
 'f1': 0.22627737226277372,
 'roc_auc': 0.8337665150530646}

In [13]:
target_names = ["no_stroke", "stroke"]

report = create_classification_report_dict(
    y_true=y_test.to_numpy(),
    y_pred=test_pred,
    target_names=target_names,
)

report

{'no_stroke': {'precision': 0.9868173258003766,
  'recall': 0.7187928669410151,
  'f1-score': 0.8317460317460318,
  'support': 729.0},
 'stroke': {'precision': 0.13135593220338984,
  'recall': 0.8157894736842105,
  'f1-score': 0.22627737226277372,
  'support': 38.0},
 'accuracy': 0.7235984354628422,
 'macro avg': {'precision': 0.5590866290018832,
  'recall': 0.7672911703126128,
  'f1-score': 0.5290117020044027,
  'support': 767.0},
 'weighted avg': {'precision': 0.9444346231189092,
  'recall': 0.7235984354628422,
  'f1-score': 0.8017488882514245,
  'support': 767.0}}

In [14]:
save_confusion_matrix_plot(
    y_true=y_test.to_numpy(),
    y_pred=test_pred,
    class_names=target_names,
    output_path=METADATA_TRIAL_001_CONFUSION_MATRIX_PATH,
)

METADATA_TRIAL_001_CONFUSION_MATRIX_PATH

WindowsPath('C:/Users/Asus/Downloads/Uni_work/Grad-Project/rural-stroke-assist/reports/figures/experiments/metadata_trial_001_sklearn_baseline_confusion_matrix.png')

In [15]:
METADATA_TRIAL_001_PREDICTIONS_PATH.parent.mkdir(parents=True, exist_ok=True)

predictions_df = test_df.copy()
predictions_df["stroke_probability"] = test_score
predictions_df["predicted_label"] = test_pred
predictions_df["true_label"] = y_test.to_numpy()
predictions_df["is_correct"] = predictions_df["predicted_label"] == predictions_df["true_label"]

predictions_df.to_csv(METADATA_TRIAL_001_PREDICTIONS_PATH, index=False)

METADATA_TRIAL_001_PREDICTIONS_PATH

WindowsPath('C:/Users/Asus/Downloads/Uni_work/Grad-Project/rural-stroke-assist/data/processed/experiments/metadata_trial_001_sklearn_baseline_predictions.csv')

In [16]:
save_markdown_results(
    metrics=test_metrics,
    report=report,
    output_path=METADATA_TRIAL_001_RESULTS_PATH,
    title=f"Metadata Trial 001 sklearn Baseline Results ({best_model_name})",
)

METADATA_TRIAL_001_RESULTS_PATH

WindowsPath('C:/Users/Asus/Downloads/Uni_work/Grad-Project/rural-stroke-assist/reports/experiments/metadata_trial_001_sklearn_baseline_results.md')

In [17]:
stroke_recall = report["stroke"]["recall"]
stroke_f1 = report["stroke"]["f1-score"]
roc_auc = test_metrics["roc_auc"]

accepted = (
    stroke_recall >= 0.60
    and roc_auc >= 0.70
)

{
    "best_model_name": best_model_name,
    "stroke_recall": stroke_recall,
    "stroke_f1": stroke_f1,
    "roc_auc": roc_auc,
    "accepted": accepted,
}

{'best_model_name': 'logistic_regression',
 'stroke_recall': 0.8157894736842105,
 'stroke_f1': 0.22627737226277372,
 'roc_auc': 0.8337665150530646,
 'accepted': True}

In [18]:
if accepted:
    save_metadata_model(best_model, METADATA_TRIAL_001_MODEL_PATH)
    print("Accepted metadata model saved to:", METADATA_TRIAL_001_MODEL_PATH)
else:
    print("Model rejected. Results and predictions saved, model artifact not saved.")

Accepted metadata model saved to: C:\Users\Asus\Downloads\Uni_work\Grad-Project\rural-stroke-assist\models\experiments\metadata\trial_001_sklearn_baseline\model.pkl
